In [6]:
import os
import shutil
import urllib.request
import torchvision

RAW_ROOT = "./data/raw"

def init_raw_folders():
    """
    只创建 raw 相关目录，避免和图片导出目录混在一起。
    """
    os.makedirs(RAW_ROOT, exist_ok=True)
    os.makedirs(os.path.join(RAW_ROOT, "svhn"), exist_ok=True)
    print(f"[Init] RAW_ROOT ready: {os.path.abspath(RAW_ROOT)}")

# ---------- MNIST (raw download via torchvision) ----------
def download_mnist():
    """
    只负责下载 MNIST 原始数据到 RAW_ROOT，不导出图片。
    download=True 会尝试拉取 4 个 gz 并解压；网络受限时可能失败。
    """
    print("\n[MNIST] Downloading to raw cache...")
    try:
        torchvision.datasets.MNIST(root=RAW_ROOT, train=True, download=True)
        torchvision.datasets.MNIST(root=RAW_ROOT, train=False, download=True)
        print("[MNIST] OK: train/test downloaded (or already present).")
        return True
    except Exception as e:
        print("[MNIST] Download failed:", repr(e))
        print("Manual fix: place these files into:")
        print(os.path.abspath(os.path.join(RAW_ROOT, "MNIST", "raw")))
        print(" - train-images-idx3-ubyte.gz")
        print(" - train-labels-idx1-ubyte.gz")
        print(" - t10k-images-idx3-ubyte.gz")
        print(" - t10k-labels-idx1-ubyte.gz")
        return False

# ---------- USPS (raw download via torchvision) ----------
def download_usps():
    """
    只负责下载 USPS 原始数据到 RAW_ROOT（通常会生成 usps.bz2 / usps.t.bz2）
    """
    print("\n[USPS] Downloading to raw cache...")
    try:
        torchvision.datasets.USPS(root=RAW_ROOT, train=True, download=True)
        torchvision.datasets.USPS(root=RAW_ROOT, train=False, download=True)
        print("[USPS] OK: train/test downloaded (or already present).")
        return True
    except Exception as e:
        print("[USPS] Download failed:", repr(e))
        print("If your network blocks it, you can manually provide:")
        print(" - usps.bz2  (train)")
        print(" - usps.t.bz2 (test)")
        print("Place them under:")
        print(os.path.abspath(RAW_ROOT))
        return False

# ---------- SVHN (manual preferred; optional auto download) ----------
def _download_file(url: str, dst_path: str) -> None:
    """
    Small helper: download a file using urllib (no extra deps).
    """
    os.makedirs(os.path.dirname(dst_path), exist_ok=True)
    print(f"  -> downloading: {url}")
    print(f"  -> to: {os.path.abspath(dst_path)}")
    urllib.request.urlretrieve(url, dst_path)

def prepare_svhn_raw(auto_download=False):
    """
    准备 SVHN 原始 .mat 文件到 RAW_ROOT/svhn/
    - 默认不自动下载（因为网络经常被拒绝，总之是404了，手动下载的http://ufldl.stanford.edu/housenumbers/）
    - auto_download=True 时会尝试 urllib 直接拉取
    """
    print("\n[SVHN] Preparing raw .mat files...")
    svhn_dir = os.path.join(RAW_ROOT, "svhn")
    train_mat = os.path.join(svhn_dir, "train_32x32.mat")
    test_mat  = os.path.join(svhn_dir, "test_32x32.mat")

    train_url = "http://ufldl.stanford.edu/housenumbers/train_32x32.mat"
    test_url  = "http://ufldl.stanford.edu/housenumbers/test_32x32.mat"

    # already exists?
    if os.path.exists(train_mat) and os.path.exists(test_mat):
        print("[SVHN] OK: train/test .mat already present.")
        return True

    if not auto_download:
        print("[SVHN] Auto download disabled (recommended under blocked network).")
        print("Please download manually and place files here:")
        print(os.path.abspath(svhn_dir))
        print("Required:")
        print(" - train_32x32.mat")
        print(" - test_32x32.mat")
        print("Sources:")
        print(" -", train_url)
        print(" -", test_url)
        return False

    # try auto download
    try:
        if not os.path.exists(train_mat):
            _download_file(train_url, train_mat)
        if not os.path.exists(test_mat):
            _download_file(test_url, test_mat)
        print("[SVHN] OK: downloaded train/test .mat.")
        return True
    except Exception as e:
        print("[SVHN] Download failed:", repr(e))
        print("Please download manually and place files here:")
        print(os.path.abspath(svhn_dir))
        return False

# ---------- entry ----------
if __name__ == "__main__":
    init_raw_folders()
    download_mnist()
    download_usps()
    # prepare_svhn_raw(auto_download=False)


Preparing MNIST dataset (offline-first)...
[MNIST] Train not found locally.
[MNIST] Trying to download train...
Using downloaded and verified file: ./data/raw\MNIST\raw\train-images-idx3-ubyte.gz
Extracting ./data/raw\MNIST\raw\train-images-idx3-ubyte.gz to ./data/raw\MNIST\raw

Using downloaded and verified file: ./data/raw\MNIST\raw\train-labels-idx1-ubyte.gz
Extracting ./data/raw\MNIST\raw\train-labels-idx1-ubyte.gz to ./data/raw\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|███████████████████████████████████████████████████████████████████| 1648877/1648877 [00:01<00:00, 1182172.74it/s]


Extracting ./data/raw\MNIST\raw\t10k-images-idx3-ubyte.gz to ./data/raw\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|████████████████████████████████████████████████████████████████████████| 4542/4542 [00:00<00:00, 12227553.77it/s]


Extracting ./data/raw\MNIST\raw\t10k-labels-idx1-ubyte.gz to ./data/raw\MNIST\raw



...

[MNIST] Train downloaded & exported.


Export test: 100%|█████████████████████████████████████████████████████████████████| 500/500 [00:00<00:00, 1268.17it/s]


[MNIST] Test exported from local cache.
Preparing USPS dataset...


USPS: 100%|██████████████████████████████████████████████████████████████████████| 2000/2000 [00:00<00:00, 2456.60it/s]

[USPS] Downloaded & exported via torchvision.


## SVHN自动下载失败，手动下载
http://ufldl.stanford.edu/housenumbers/

Yuval Netzer, Tao Wang, Adam Coates, Alessandro Bissacco, Bo Wu, Andrew Y. Ng Reading Digits in Natural Images with Unsupervised Feature Learning NIPS Workshop on Deep Learning and Unsupervised Feature Learning 2011. (PDF)

Please use http://ufldl.stanford.edu/housenumbers as the URL for this site when necessary

In [10]:
import os
import shutil
import scipy.io as sio
from PIL import Image
import torchvision
from tqdm import tqdm

# 所有原始数据现在都统一存放在 data/raw 及其子目录下
RAW_ROOT = './data/raw'

def init_split_folders():
    # 我们只清理和初始化用于存放“提取后图片”的目录
    # 这些目录与存储 .mat/.bz2 的 raw 目录是完全分开的
    extract_dirs = ['data/mnist', 'data/svhn', 'data/usps']
    for d in extract_dirs:
        if os.path.exists(d):
            # 这里只会删除生成的图片文件夹，不会触及 data/raw
            shutil.rmtree(d) 
        for split in ['train', 'test']:
            os.makedirs(os.path.join(d, split), exist_ok=True)
    print("图片提取目录 (train/test) 已初始化，原始数据区已受保护。")

def extract_svhn(train_num=2000, test_num=500):
    print("正在提取 SVHN...")
    # 修正后的原始文件路径：data/raw/svhn/
    svhn_raw_dir = os.path.join(RAW_ROOT, 'svhn')
    
    try:
        train_mat = sio.loadmat(os.path.join(svhn_raw_dir, 'train_32x32.mat'))
        test_mat = sio.loadmat(os.path.join(svhn_raw_dir, 'test_32x32.mat'))
        
        for split, mat, num in [('train', train_mat, train_num), ('test', test_mat, test_num)]:
            X, y = mat['X'], mat['y']
            for i in tqdm(range(num), desc=f"SVHN {split.capitalize()}"):
                img = X[:, :, :, i]
                label = int(y[i][0])
                label = 0 if label == 10 else label # 修正标签 10 -> 0
                # 图片保存在 data/svhn/ 下，不会覆盖 data/raw/svhn/ 里的原始文件
                save_path = f'data/svhn/{split}/{i:05d}_label_{label}.png'
                Image.fromarray(img).save(save_path)
    except FileNotFoundError:
        print(f"错误：找不到 SVHN 原始文件。请确保 .mat 文件位于 {svhn_raw_dir}")

# extract_mnist 和 extract_usps 逻辑保持不变，确保 root 使用 RAW_ROOT
def extract_mnist(train_num=2000, test_num=500):
    print("正在提取 MNIST...")
    train_set = torchvision.datasets.MNIST(root=RAW_ROOT, train=True, download=False)
    test_set = torchvision.datasets.MNIST(root=RAW_ROOT, train=False, download=False)
    for i in tqdm(range(train_num), desc="MNIST Train"):
        img, label = train_set[i]
        img.save(f'data/mnist/train/{i:05d}_label_{label}.png')
    for i in tqdm(range(test_num), desc="MNIST Test"):
        img, label = test_set[i]
        img.save(f'data/mnist/test/{i:05d}_label_{label}.png')

def extract_usps(train_num=2000, test_num=500):
    print("正在提取 USPS...")
    # 自动处理 usps.t.bz2 逻辑
    train_file = os.path.join(RAW_ROOT, "usps.bz2")
    test_file = os.path.join(RAW_ROOT, "usps.t.bz2")
    if os.path.exists(train_file) and not os.path.exists(test_file):
        shutil.copy(train_file, test_file)

    train_set = torchvision.datasets.USPS(root=RAW_ROOT, train=True, download=False)
    test_set = torchvision.datasets.USPS(root=RAW_ROOT, train=False, download=False)
    for i in tqdm(range(train_num), desc="USPS Train"):
        img, label = train_set[i]
        img.save(f'data/usps/train/{i:05d}_label_{label}.png')
    for i in tqdm(range(test_num), desc="USPS Test"):
        img, label = test_set[i]
        img.save(f'data/usps/test/{i:05d}_label_{label}.png')

if __name__ == "__main__":
    init_split_folders()
    extract_mnist()
    extract_usps()
    extract_svhn()

图片提取目录 (train/test) 已初始化，原始数据区已受保护。
正在提取 MNIST...


MNIST Test: 100%|██████████████████████████████████████████████████████████████████| 500/500 [00:00<00:00, 1299.26it/s]


正在提取 USPS...


USPS Test: 100%|███████████████████████████████████████████████████████████████████| 500/500 [00:00<00:00, 1451.20it/s]


正在提取 SVHN...


SVHN Test: 100%|███████████████████████████████████████████████████████████████████| 500/500 [00:00<00:00, 1007.57it/s]
